<!-- nav -->
[← 11 · A parts catalog: schemas that build on schemas](11_A_Parts_Catalog.ipynb) · [Home](../../README.md) · [TypeScript tutorial →](../../typescript5/tutorials/README.md)

# 12 · Keeping history: provenance, journals and snapshots

**The problem.** A team discusses its work on a board of comments. Someone asks who wrote a comment, when, and from
where, and who changed it last. A second service must hold the same comments, written by the same transactions. And a
restart must bring everything back as it was, history included, without adding history of its own.

Three things answer these:

1. **Provenance**, an extension, records each transaction that writes: who, when, in which session and where, and,
   for the schemas the app chooses, which objects each transaction created and which it modified last
   ([CAP-134: transactions recorded with provenance](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-134.md)).
2. A **journal** is a list of transactions' expressions, written as data; another store replays it by executing them,
   and makes the same objects and the same records
   ([CAP-129: journals replayed by execution](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-129.md)).
3. A **snapshot** writes a whole store, its schemas and its data, as two transactions, which a store reads back
   ([CAP-127: snapshots made of transactions](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-127.md)).

In [1]:
import re

from mbse.Schemas.Framework import JSON, Builders, Plain, Provenance, References


def install(txn):
    """A team's board, a singleton owning its comments, each given its ID by the board."""
    S = txn.Schemas  # the store's own schemas, by name
    return txn.uid('tutorial.board/1').mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({'Board': S.OfObject, 'Posted': S.OfRelation, 'Comment': S.OfObject})
        .spec([
            lambda _: _.Posted.name('work.Posted').links([lambda t: t.name('board'), lambda t: t.name('comment')])
            .properties([lambda t: t.name('id').schema([S.OfStr]).pattern(r'C-(\d+)')])
            .uniques([['board', 'id'], ['comment']]),
            lambda _: _.Board.name('work.Board').ref(True).singleton('work.TheBoard').adjacencies({
                'comments': lambda a: a.relation([_.Posted]).me('board')}),
            lambda _: _.Comment.name('work.Comment').ref(True).properties({'text': S.OfStr}).adjacencies([
                lambda t: t.name('board').relation([_.Posted]).me('comment').owner(True)]),
        ])]))])

The board owns each comment through `work.Posted`, whose `id` has a naming pattern (case study 6): a comment built
anywhere is placed on the board, the one singleton that can own it, under the next ID, `C-1`, `C-2`. An owned object
has a path, so a transaction in another store, or on another day, can refer to it.

<a id="T12.1"></a>
## 12.1 · Who changed what: provenance

`Provenance` is installed after the app's own install, and `Provenance.extend` names the schemas whose objects it
links to their records, here comments. A transaction then says who writes, when and from where: `when` is UTC ISO 8601
text, the session has an id and the identity it belongs to (`who`), a locale and a device, and `where` a latitude and
longitude. A service sets them from the request it serves; the record is made from them, so that a replay makes the
same one. A transaction built first, with `store.Mutations.OfTransaction`, is run as any is:

In [2]:
def made():
    """A store of the board, recording the history of its comments."""
    store = Builders.Store()
    store.transaction(install)
    store.transaction(Provenance.install)  # after the app's own install
    store.transaction(lambda t: Provenance.extend(t, [t.work.Comment]))  # the schemas whose history it keeps
    return store


store = made()
board = store.singleton('work.TheBoard')


def post(uid, when, text):
    """A transaction posting a comment, as a service writes it: who, when and where, from the request."""
    return store.Mutations.OfTransaction(lambda t: t.uid(uid).when(when)
        .session(lambda s: s.id('tab-7f3a').who('ada').locale('en-GB'))
        .where(lambda w: w.latitude(51.5).longitude(-0.125))
        .mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
            .symbols({'c': store.work.Comment})
            .spec([lambda _: _.c.text(text)])]))]))


first = post('post-1', '2026-10-26T14:30:05Z', 'Should chips show adjacencies?')
comment = store.transaction(first)[0]['c']
print([(str(e.id), str(e.comment.text)) for e in board.comments])

[('C-1', 'Should chips show adjacencies?')]


The records are data in the same store, each in a directory singleton keyed by name: transactions by their `uid`,
sessions by their id, identities by their name. The record holds what the transaction said:

In [3]:
def keys(name, adjacency, key):
    """The keys of a directory singleton's entries."""
    return sorted(str(getattr(e, key)) for e in getattr(store.singleton(name), adjacency))


record = next(e.transaction for e in store.singleton('Provenance.Transactions').transactions if str(e.uid) == 'post-1')
where = store.member(record, 'where')
print(keys('Provenance.Transactions', 'transactions', 'uid'), str(record.when),
      (store.member(where, 'latitude'), store.member(where, 'longitude')))
print(keys('Provenance.Sessions', 'sessions', 'id'), keys('Provenance.Identities', 'identities', 'name'))

['post-1'] 2026-10-26T14:30:05Z (51.5, -0.125)
['tab-7f3a'] ['ada']


<a id="T12.2"></a>
## 12.2 · An object's records, read from the record's side

Ada edits her comment in a second transaction, which refers to it by its reference (case study 10). The links from a
comment to its records, `CreatedBy` and `ModifiedBy`, are anonymous adjacencies: a comment has no member for them, so
its schema, and every program that reads it, is unchanged
([CAP-178: an object's provenance read through its own named adjacencies](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-178.md) plans named ones). Read
them from the record's side, with a map (case study 10) whose symbol is given the comment's reference: the record that
`created` it, its session and the session's identity; and the record that `modified` it, the latest write only.

In [4]:
def edit(uid, when, text):
    """A transaction writing a comment's text, the comment given by its reference."""
    return store.Mutations.OfTransaction(lambda t: t.uid(uid).when(when)
        .session(lambda s: s.id('tab-7f3a').who('ada'))
        .mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
            .symbols({'c': References.of(store, comment)})
            .spec([lambda _: _.c.text(text)])]))]))


second = edit('edit-2', '2026-10-26T15:00:00Z', 'Should chips show their adjacencies?')
store.transaction(second)
created = store.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p
    .symbols({'c': References.of(store, comment), 't': store.Provenance.OfTransaction,
              's': store.Provenance.OfSession, 'i': store.Provenance.OfIdentity})
    .spec([lambda _: _.t.created([_.c]), lambda _: _.t.session([_.s]), lambda _: _.s.who([_.i])])]))]))
modified = store.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p
    .symbols({'c': References.of(store, comment), 't': store.Provenance.OfTransaction})
    .spec([lambda _: _.t.modified([_.c])])]))]))
frame = created[0]  # one frame: the record, its session and the identity
print(len(created), str(frame['t'].when), str(frame['s'].locale), [str(e.name) for e in frame['i'].directory])
print([[str(e.uid) for e in f['t'].directory] for f in modified])

1 2026-10-26T14:30:05Z en-GB ['ada']
[['edit-2']]


A write made imperatively, within `with store.transaction():`, has no expression to say who made it. Its record
holds placeholders: a random UUID for its uid, the clock's time, and the session and identity `unknown`. A service
that knows its caller writes an expression instead.

In [5]:
with store.transaction():
    store.work.Comment(comment, lambda c: c.text('Should chips show their adjacencies, Bob?'))
uids = keys('Provenance.Transactions', 'transactions', 'uid')
uuid = re.compile(r'[0-9a-f]{8}-[0-9a-f]{4}-4[0-9a-f]{3}-[89ab][0-9a-f]{3}-[0-9a-f]{12}')
print(len(uids), [u for u in uids if not uuid.fullmatch(u)], keys('Provenance.Sessions', 'sessions', 'id'))

3 ['edit-2', 'post-1'] ['tab-7f3a', 'unknown']


<a id="T12.3"></a>
## 12.3 · A journal, replayed in another store

A transaction's expression is data: `Plain.ToPlain(store).Transaction(tx)` writes it, and
`Plain.FromPlain(other).Transaction(data)` reads it into a store, to run there. A **journal** is a list of them, kept
in order. The second service makes a store of the same installs and replays the journal: each expression finds what it
names (the comment by its reference, a path) or makes it, and its record is made from what it says, so the records are
the same. Replayed again, an expression writes nothing twice: the upsert finds what it made, and the record its uid.

In [6]:
journal = [JSON.dumps(Plain.ToPlain(store).Transaction(tx)) for tx in (first, second)]  # as text, to send or keep
print(journal[1][:220], '...')

again = made()
for entry in journal:
    again.transaction(Plain.FromPlain(again).Transaction(JSON.loads(entry)))
    again.transaction(Plain.FromPlain(again).Transaction(JSON.loads(entry)))  # replayed: nothing written twice
print([(str(e.id), str(e.comment.text)) for e in again.singleton('work.TheBoard').comments])
print(sorted(str(e.uid) for e in again.singleton('Provenance.Transactions').transactions))

{"uid": "edit-2", "when": "2026-10-26T15:00:00Z", "session": {"id": "tab-7f3a", "who": "ada"}, "mutations": [{"upsert": {"exists": [{"symbols": {"c": {"reference": {"path": {"singleton": "work.TheBoard", "next": {"entry" ...
[('C-1', 'Should chips show their adjacencies?')]
['edit-2', 'post-1']


Replay a journal in its order. An upsert finds an owned object by its spec, so replaying the first post after the
edit, whose text it no longer matches, would make a second comment: what an upsert finds or makes when run again
depends on what the store holds then
([CAP-139: upserts find before they create](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-139.md)).

<a id="T12.4"></a>
## 12.4 · A snapshot, and a restart

`Plain.ToPlain(store).Snapshot()`, and its JSON and YAML forms, write a whole store as two transactions: its schemas,
each in its module, and then the data its singletons reach, the provenance records included. Reading it back executes
them: a schema the reading store holds is found by its name, and the rest are made. A restart writes a snapshot and
reads it into a new store; the records come back, and reading adds none, since what a snapshot writes is the history it
holds, not a change of its own. Written again, the snapshot is the same.

In [7]:
text = JSON.ToJSON(store).Snapshot()
restarted = made()
JSON.FromJSON(restarted).Snapshot(text)
print([str(e.comment.text) for e in restarted.singleton('work.TheBoard').comments])
print(sorted(str(e.uid) for e in restarted.singleton('Provenance.Transactions').transactions) == uids)
print(JSON.ToJSON(restarted).Snapshot() == text)

['Should chips show their adjacencies, Bob?']
True


True


## What you learned

- `Provenance`, installed after the app's own install, records each transaction that writes; `Provenance.extend`
  links the objects of the schemas it names to the records that created them and that modified them last.
- A transaction's expression gives the record: `uid`, `when`, `session` (`id`, `who`, `locale`, `device`) and
  `where`. An imperative write's record holds placeholders.
- An object's records are read from the record's side, with a map whose symbol is given the object's reference.
- A journal is a list of expressions, written and read with `Plain`'s `Transaction`; replayed in order in another
  store, it makes the same objects and records, and replayed again, nothing twice.
- A snapshot writes a whole store, schemas and data, as two transactions; read back, it restores the records and adds
  none.

**Next:** the [TypeScript tutorial](../../typescript5/tutorials/README.md) tells the same twelve case studies, and
[mbse-expressions](https://github.com/pitaman71/mbse-expressions) continues where case study 10 stops.

<!-- nav -->
[← 11 · A parts catalog: schemas that build on schemas](11_A_Parts_Catalog.ipynb) · [Home](../../README.md) · [TypeScript tutorial →](../../typescript5/tutorials/README.md)